[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-3-ai-agents/adv-08-agent-evaluation-statistics.ipynb)

# Advanced Discussion 8 — intervals, pass@k versus pass^k, power and judge bias

Wilson intervals, repeated-attempt metrics, a power simulation for comparing two agents, and position bias in an LLM judge.

**Optional advanced-discussion lab** for AI Agents and Agentic AI with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier; CPU is enough unless a note says otherwise.

Simulation only; the power simulation takes a minute or two.

In [ ]:
%pip install -q numpy

In [ ]:
import math, numpy as np
rng = np.random.RandomState(0)

## 1. confidence intervals for a pass rate on a small task suite

In [ ]:
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n); w = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)); return (c - w) / d, (c + w) / d
def normal_ci(k, n, z=1.96):
    p = k / n; w = z * math.sqrt(p * (1 - p) / n); return max(0, p - w), min(1, p + w)
print("observed pass rate           Wilson 95% interval      normal-approximation interval")
for k, n in ((18, 20), (20, 20), (36, 40), (45, 50), (90, 100), (270, 300)):
    print("%3d / %3d = %.2f            [%.2f, %.2f]              [%.2f, %.2f]" % (k, n, k / n, *wilson(k, n), *normal_ci(k, n)))
print("width of the Wilson interval at a 90%% pass rate: n=20 %.2f | n=50 %.2f | n=100 %.2f | n=300 %.2f" % tuple(np.diff(wilson(int(0.9 * n), n))[0] for n in (20, 50, 100, 300)))

## 2. pass@k vs pass^k: one lucky run is not reliability

In [ ]:
# agents solve each task with its own probability (some tasks are easy, some hard); run each task k times
p_task = rng.beta(4, 1.5, 2000)                                                # mean ~0.73 with a spread
print("\nper-task success probability drawn from Beta(4, 1.5): mean %.3f" % p_task.mean())
print("  k    pass@k (at least one of k succeeds)   pass^k (ALL k succeed)")
for k in (1, 2, 4, 8): print("%3d    %.3f                                %.3f" % (k, np.mean(1 - (1 - p_task) ** k), np.mean(p_task ** k)))
print("  an agent that looks like %.0f%% on one try is %.0f%% reliable if the customer needs it to work 8 times running" % (100 * p_task.mean(), 100 * np.mean(p_task ** 8)))

## 3. how many tasks do you need to tell two agents apart? (paired, same tasks)

In [ ]:
def detect(n, pa, pb, rho=0.7, sims=600, seed=1):
    r = np.random.RandomState(seed); hits = 0
    for _ in range(sims):
        u = r.rand(n); v = np.where(r.rand(n) < rho, u, r.rand(n)); a = u < pa; b = v < pb
        d = b.astype(float) - a.astype(float); boots = [d[r.randint(0, n, n)].mean() for _ in range(200)]
        hits += np.percentile(boots, 2.5) > 0
    return hits / sims
print("\nagent B truly 8 points better (0.70 vs 0.78), errors correlated: share of experiments whose 95% interval excludes zero")
for n in (25, 50, 100, 200, 400): print("  %4d tasks: %.0f%%" % (n, 100 * detect(n, 0.70, 0.78)))

## 4. a judge that prefers the first answer it sees

In [ ]:
def judge_with_position_bias(n, true_pref_b=0.5, bias=0.10, seed=2):
    r = np.random.RandomState(seed); wins_b_first = 0; wins_b_second = 0
    for _ in range(n):
        p_b_first = min(1, true_pref_b + bias); p_b_second = max(0, true_pref_b - bias)
        wins_b_first += r.rand() < p_b_first; wins_b_second += r.rand() < p_b_second
    return wins_b_first / n, wins_b_second / n, (wins_b_first + wins_b_second) / (2 * n)
f, s, avg = judge_with_position_bias(1000)
print("\ntwo equally good answers, judge with a 10-point bias for whichever is shown first: B wins %.2f when shown first, %.2f when second; averaging both orders gives %.2f" % (f, s, avg))

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.